# vrs-matcher — Example Cohort

This notebook walks through the full vrs-matcher workflow on a small synthetic
cohort bundled with the repository:

1. **Install** the package and dependencies
2. **Download** the example VCF from GitHub
3. **Load** samples into a local SQLite index
4. **Inspect** the index
5. **Match** a pair of samples
6. **Rank** one sample against all others
7. **Visualize** the full pairwise similarity matrix

Each step shows both the **Python API** and the equivalent **CLI command**,
so you can see how the two relate.

The example VCF contains three samples (`SAMPLE_A`, `SAMPLE_B`, `SAMPLE_C`) at
three biallelic sites, each pre-annotated with [GA4GH VRS](https://www.ga4gh.org/product/variation-representation/) allele identifiers.

## 1 · Install

In [ ]:
# cyvcf2 ships a pre-built wheel for the Colab Linux runtime.
# vrs-matcher is installed directly from the GitHub repository.
%pip install --quiet cyvcf2 matplotlib numpy
%pip install --quiet git+https://github.com/EllrottLab/vrs-matcher.git

## 2 · Download the example VCF

In [ ]:
import urllib.request
from pathlib import Path

BASE = "https://raw.githubusercontent.com/EllrottLab/vrs-matcher/refs/heads/development/examples"
FILES = [
    "example_cohort.vcf.gz",
    "example_cohort.vcf.gz.tbi",
]

for fname in FILES:
    dest = Path(fname)
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{fname}", dest)
    print(f"✓ {dest}  ({dest.stat().st_size:,} bytes)")

VCF_PATH = "example_cohort.vcf.gz"

### Peek at the VCF header and records

In [ ]:
import cyvcf2

vcf = cyvcf2.VCF(VCF_PATH)
print("Samples:", vcf.samples)
print()
for rec in vcf:
    vrs = rec.INFO.get("VRS_Allele_IDs", "—")
    print(f"{rec.CHROM}:{rec.POS}  REF={rec.REF}  ALT={rec.ALT}  VRS={vrs}")
vcf.close()

## 3 · Load samples into the index

### Python API

In [ ]:
from vrs_matcher.loader import load_samples

DB_PATH = "cohort.db"

n = load_samples(VCF_PATH, DB_PATH)
print(f"Loaded {n} allele record(s) into {DB_PATH}")

### CLI equivalent

In [ ]:
!vrs-matcher load-samples example_cohort.vcf.gz --db cohort.db

## 4 · Inspect the index

In [ ]:
from vrs_matcher.db import open_db

conn = open_db(DB_PATH)

print("=== Registered samples ===")
for row in conn.execute("SELECT sample_id FROM samples ORDER BY sample_id"):
    print(" ", row["sample_id"])

print()
print("=== Allele index (sample_allele) ===")
header = f"{'sample_id':<12} {'vrs_id':<28} {'gt':<6} {'zygosity':<10} {'chrom':<6} {'pos'}"
print(header)
print("-" * len(header))
for row in conn.execute(
    "SELECT sample_id, vrs_id, gt, zygosity, chrom, pos FROM sample_allele ORDER BY chrom, pos, sample_id"
):
    print(f"{row['sample_id']:<12} {row['vrs_id']:<28} {row['gt']:<6} {row['zygosity']:<10} {row['chrom']:<6} {row['pos']}")

conn.close()

## 5 · Match a pair of samples

### Python API

In [ ]:
from vrs_matcher.db import open_db
from vrs_matcher.matcher import match_pair

conn = open_db(DB_PATH)

result = match_pair(conn, "SAMPLE_A", "SAMPLE_B")

print(f"SAMPLE_A  vs  SAMPLE_B")
print(f"  Jaccard similarity:   {result.jaccard:.4f}")
print(f"  Weighted concordance: {result.weighted_concordance:.4f}")
print(f"  Shared VRS IDs:       {len(result.shared_vrs_ids)}")
print(f"  Total alleles (A/B):  {result.total_a} / {result.total_b}")
if result.shared_vrs_ids:
    print(f"  Shared IDs:")
    for vid in sorted(result.shared_vrs_ids):
        print(f"    {vid}")

conn.close()

### CLI equivalent

In [ ]:
!vrs-matcher match-samples SAMPLE_A SAMPLE_B --db cohort.db

## 6 · Rank SAMPLE_A against all others

### Python API

In [ ]:
from vrs_matcher.matcher import match_against_all

conn = open_db(DB_PATH)
results = match_against_all(conn, "SAMPLE_A")
conn.close()

print(f"{'Rank':<5} {'Sample':<12} {'Jaccard':>8} {'WConc':>8} {'Shared':>8}")
print("-" * 46)
for i, r in enumerate(results, 1):
    other = r.sample_b if r.sample_a == "SAMPLE_A" else r.sample_a
    print(f"{i:<5} {other:<12} {r.jaccard:>8.4f} {r.weighted_concordance:>8.4f} {len(r.shared_vrs_ids):>8}")

### CLI equivalent

In [ ]:
!vrs-matcher match-sample SAMPLE_A --db cohort.db

## 7 · Pairwise similarity heatmap

In [ ]:
import itertools
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

conn = open_db(DB_PATH)

from vrs_matcher.db import list_samples

samples = sorted(list_samples(conn))
n = len(samples)
idx = {s: i for i, s in enumerate(samples)}

# Build symmetric Jaccard matrix (diagonal = 1.0)
matrix = np.eye(n)
for a, b in itertools.combinations(samples, 2):
    r = match_pair(conn, a, b)
    matrix[idx[a], idx[b]] = r.jaccard
    matrix[idx[b], idx[a]] = r.jaccard

conn.close()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(matrix, vmin=0, vmax=1, cmap="YlOrRd")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(samples, rotation=30, ha="right", fontsize=9)
ax.set_yticklabels(samples, fontsize=9)
ax.set_title("Pairwise Jaccard similarity", fontsize=11)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=10,
                color="black" if matrix[i, j] < 0.7 else "white")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()